In [16]:
import sqlite3
import pandas as pd
import os

In [ ]:
INPUT_PATH = "/Users/kamaleshwari/Desktop/decoding customer value/modified_dataset.csv"
OUTPUT_DIR = "/Users/kamaleshwari/Desktop/decoding customer value/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(INPUT_PATH)

In [18]:
con = sqlite3.connect(":memory:")
df.to_sql("customers", con, index=False, if_exists="replace")

def query(sql):
    """Run a SQL query and return a DataFrame."""
    return pd.read_sql_query(sql, con)

print(f"Loaded {len(df):,} customers into SQLite (in-memory)")
print("Segment distribution:")
print(df["customer_segment"].value_counts().to_string())

Loaded 3,900 customers into SQLite (in-memory)
Segment distribution:
customer_segment
Passive Buyer          1830
Subscription Member    1053
Promo Hunter            624
Brand Advocate          393


In [ ]:
#Q1- who are the customers likely to still buy after two years from now on, and what would they look like today?
# Classifies every customer into one of four loyalty archetypes using a
# priority cascade anchored on two competing loyalty definitions:
#   Organically Loyal       = loyalty_def2 (high-freq + satisfied + no promo)
#   Subscription-Committed  = loyalty_def1 only (historical, always with promo)
#   Promo-Only Buyer        = neither definition, uses promos
#   Occasional Buyer        = neither definition, no promos
Q1 = """
SELECT
    loyalty_type,
    n_customers,
    ROUND(n_customers * 100.0 / SUM(n_customers) OVER (), 1)    AS pct_of_base,
    avg_clv,
    total_clv,
    ROUND(total_clv * 100.0 / SUM(total_clv) OVER (), 1)        AS pct_total_clv,
    avg_spend,
    avg_prev_purchases,
    avg_rating,
    avg_freq_score
FROM (
    SELECT
        CASE
            WHEN loyalty_def2 = 1                        THEN 'Organically Loyal'
            WHEN loyalty_def1 = 1 AND loyalty_def2 = 0   THEN 'Subscription-Committed'
            WHEN promo_dependency_score = 1              THEN 'Promo-Only Buyer'
            ELSE                                              'Occasional Buyer'
        END                                              AS loyalty_type,
        COUNT(*)                                         AS n_customers,
        ROUND(AVG(clv_proxy), 2)                         AS avg_clv,
        ROUND(SUM(clv_proxy), 2)                         AS total_clv,
        ROUND(AVG(purchase_amount), 2)                   AS avg_spend,
        ROUND(AVG(previous_purchases), 1)                AS avg_prev_purchases,
        ROUND(AVG(review_rating), 2)                     AS avg_rating,
        ROUND(AVG(frequency_score), 2)                   AS avg_freq_score
    FROM customers
    GROUP BY loyalty_type
) sub
ORDER BY avg_clv DESC
"""
 
q1 = query(Q1)
print("Q1")
print(q1.to_string(index=False))

q1.to_csv(f"{OUTPUT_DIR}/q1_loyalty_archetypes.csv", index=False)
 

Q1
          loyalty_type  n_customers  pct_of_base  avg_clv  total_clv  pct_total_clv  avg_spend  avg_prev_purchases  avg_rating  avg_freq_score
     Organically Loyal          393         10.1  1111.15  436681.19           15.6      60.04                25.7        4.46            5.99
Subscription-Committed          569         14.6   873.57  497062.41           17.8      59.00                37.3        3.75            4.10
      Promo-Only Buyer         1108         28.4   639.36  708410.54           25.4      59.42                19.8        3.74            3.90
      Occasional Buyer         1830         46.9   629.36 1151728.09           41.2      60.15                24.9        3.61            3.51


In [ ]:
#Q2 - Is the discount and promo program actually building a loyal customer base or just attracting one-time bargain hunters?
#value-tier behavioral profiles
Q2a = """
SELECT
    value_tier,
    COUNT(*)                                            AS n_customers,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(frequency_score), 2)                      AS avg_freq_score,
    ROUND(AVG(previous_purchases), 2)                   AS avg_prev_purchases,
    ROUND(AVG(review_rating), 2)                        AS avg_rating,
    ROUND(AVG(subscription_status_flag) * 100, 1)       AS pct_subscribed,
    ROUND(AVG(promo_dependency_score) * 100, 1)         AS pct_promo,
    ROUND(AVG(satisfaction_flag) * 100, 1)              AS pct_satisfied
FROM customers
GROUP BY value_tier
ORDER BY avg_clv DESC
"""
 
q2a = query(Q2a)
print("Q2a — VALUE TIER BEHAVIORAL PROFILES")
print(q2a.to_string(index=False))
q2a.to_csv(f"{OUTPUT_DIR}/q2a_value_tier_profiles.csv", index=False)

Q2a — VALUE TIER BEHAVIORAL PROFILES
value_tier  n_customers  avg_clv  avg_freq_score  avg_prev_purchases  avg_rating  pct_subscribed  pct_promo  pct_satisfied
   Premium          974  1484.30            5.77               31.75        3.77            29.0       44.1           43.5
      High          976   768.95            4.46               26.70        3.74            26.4       41.5           41.9
       Mid          974   431.30            3.63               23.75        3.73            27.2       44.5           41.3
       Low          976   181.96            1.96               19.22        3.76            25.4       41.9           40.9


In [ ]:
#RFM SEGMENT × VALUE TIER CROSS-TAB
Q2b = """
SELECT
    rfm_segment,
    ROUND(SUM(CASE WHEN value_tier = 'Premium' THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                AS pct_premium,
    ROUND(SUM(CASE WHEN value_tier = 'High'    THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                AS pct_high,
    ROUND(SUM(CASE WHEN value_tier = 'Mid'     THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                AS pct_mid,
    ROUND(SUM(CASE WHEN value_tier = 'Low'     THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                AS pct_low,
    COUNT(*)                                            AS n_customers,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv
FROM customers
GROUP BY rfm_segment
ORDER BY avg_clv DESC
"""
 
q2b = query(Q2b)
print("Q2b — RFM SEGMENT → VALUE TIER CROSS-TAB")
print(q2b.to_string(index=False))
q2b.to_csv(f"{OUTPUT_DIR}/q2b_rfm_vs_value_tier.csv", index=False)

Q2b — RFM SEGMENT → VALUE TIER CROSS-TAB
    rfm_segment  pct_premium  pct_high  pct_mid  pct_low  n_customers  avg_clv
      Champions         99.7       0.3      0.0      0.0          307  1862.72
          Loyal         48.9      39.6     10.8      0.8         1328  1013.06
Needs Attention          1.1      27.1     45.0     26.8         1659   463.55
        At-Risk          0.0       0.0     14.0     86.0          606   177.65


In [ ]:
# Q2c — PURCHASE FREQUENCY → CLV
Q2c = """
SELECT
    frequency_of_purchases,
    frequency_score,
    COUNT(*)                                            AS n,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(purchase_amount), 2)                      AS avg_spend,
    ROUND(AVG(previous_purchases), 1)                   AS avg_prev_purchases,
    ROUND(AVG(promo_dependency_score) * 100, 1)         AS pct_promo,
    ROUND(AVG(satisfaction_flag) * 100, 1)              AS pct_satisfied
FROM customers
GROUP BY frequency_of_purchases, frequency_score
ORDER BY frequency_score DESC
"""
 
q2c = query(Q2c)
print("Q2c — PURCHASE FREQUENCY → CLV PROFILE")
print(q2c.to_string(index=False))
q2c.to_csv(f"{OUTPUT_DIR}/q2c_frequency_vs_clv.csv", index=False)

Q2c — PURCHASE FREQUENCY → CLV PROFILE
frequency_of_purchases  frequency_score   n  avg_clv  avg_spend  avg_prev_purchases  pct_promo  pct_satisfied
                Weekly                7 539  1256.55      58.97                25.8       43.4           42.3
             Bi-Weekly                6 547  1096.43      60.69                24.8       41.3           41.3
           Fortnightly                5 542   900.36      59.05                25.3       44.8           41.9
               Monthly                4 553   719.92      59.33                25.3       42.9           44.3
        Every 3 Months                3 584   542.94      60.08                25.0       44.2           42.6
             Quarterly                2 563   373.04      59.98                26.9       42.8           39.4
              Annually                1 572   181.20      60.17                24.6       41.6           41.4


In [ ]:
# Q2d — SEASON × CATEGORY × PURCHASE TENURE 
# PS explicitly requires: "which seasons and categories are associated with
# lower-tenure customers versus those with high previous purchase counts."
# Tenure bands anchored at quartiles: Q1=13, median=25, Q3=38.
Q2d = """
SELECT
    season,
    category,
    COUNT(*)                                            AS n_customers,
    ROUND(AVG(previous_purchases), 1)                   AS avg_prev_purchases,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(frequency_score), 2)                      AS avg_freq_score,
    ROUND(AVG(promo_dependency_score) * 100, 1)         AS pct_promo,
    ROUND(AVG(satisfaction_flag) * 100, 1)              AS pct_satisfied,
    CASE
        WHEN AVG(previous_purchases) >= 27 THEN 'High-Tenure  (>=27 purchases)'
        WHEN AVG(previous_purchases) >= 23 THEN 'Mid-Tenure   (23-26 purchases)'
        ELSE                                    'Low-Tenure   (<23 purchases)'
    END                                                 AS tenure_band
FROM customers
GROUP BY season, category
ORDER BY avg_prev_purchases DESC
"""
 
q2d = query(Q2d)
print("Q2d — SEASON x CATEGORY x PURCHASE TENURE  [PS-required]")
print(q2d.to_string(index=False))
q2d.to_csv(f"{OUTPUT_DIR}/q2d_season_category_tenure.csv", index=False)

Q2d — SEASON x CATEGORY x PURCHASE TENURE  [PS-required]
season    category  n_customers  avg_prev_purchases  avg_clv  avg_freq_score  pct_promo  pct_satisfied                    tenure_band
Summer Accessories          312                26.8   716.55            3.82       42.6           43.3 Mid-Tenure   (23-26 purchases)
Winter    Footwear          140                26.7   788.33            4.06       40.0           44.3 Mid-Tenure   (23-26 purchases)
Winter    Clothing          448                26.1   753.26            4.04       42.6           39.7 Mid-Tenure   (23-26 purchases)
Spring   Outerwear           81                26.0   682.54            3.96       40.7           42.0 Mid-Tenure   (23-26 purchases)
Spring Accessories          301                25.9   693.45            4.05       46.5           46.5 Mid-Tenure   (23-26 purchases)
Summer    Footwear          160                25.6   725.53            3.94       45.6           41.9 Mid-Tenure   (23-26 purchases)
  Fal

In [ ]:
# Q2e — FREQUENCY × TENURE INTERACTION 
# PS requires patterns among "high-frequency, high-tenure customers."
# Tenure quartiles: Top (38-50), Mid (25-37), Lower (13-24), New (1-12).
Q2e = """
SELECT
    frequency_of_purchases,
    CASE
        WHEN previous_purchases >= 38 THEN 'Top-Tenure    (38-50)'
        WHEN previous_purchases >= 25 THEN 'Mid-Tenure    (25-37)'
        WHEN previous_purchases >= 13 THEN 'Lower-Tenure  (13-24)'
        ELSE                               'New Customer  (1-12)'
    END                                                 AS tenure_quartile,
    COUNT(*)                                            AS n_customers,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(purchase_amount), 2)                      AS avg_spend,
    ROUND(AVG(promo_dependency_score) * 100, 1)         AS pct_promo,
    ROUND(AVG(satisfaction_flag) * 100, 1)              AS pct_satisfied,
    ROUND(AVG(review_rating), 2)                        AS avg_rating
FROM customers
GROUP BY frequency_of_purchases, tenure_quartile
ORDER BY avg_clv DESC
"""
 
q2e = query(Q2e)
print("Q2e — FREQUENCY x TENURE INTERACTION  [PS-required]")
print(q2e.to_string(index=False))
q2e.to_csv(f"{OUTPUT_DIR}/q2e_frequency_tenure_interaction.csv", index=False)

Q2e — FREQUENCY x TENURE INTERACTION  [PS-required]
frequency_of_purchases       tenure_quartile  n_customers  avg_clv  avg_spend  pct_promo  pct_satisfied  avg_rating
                Weekly Top-Tenure    (38-50)          134  1587.71      59.40       46.3           46.3        3.87
                Weekly Mid-Tenure    (25-37)          156  1413.06      58.27       45.5           41.7        3.72
             Bi-Weekly Top-Tenure    (38-50)          140  1371.10      60.00       42.1           45.0        3.75
             Bi-Weekly Mid-Tenure    (25-37)          129  1265.41      61.24       42.6           42.6        3.72
                Weekly Lower-Tenure  (13-24)          125  1234.96      59.61       41.6           43.2        3.71
           Fortnightly Top-Tenure    (38-50)          134  1155.72      60.87       50.0           41.8        3.78
             Bi-Weekly Lower-Tenure  (13-24)          142  1067.15      60.49       41.5           38.0        3.67
           Fortnight

In [ ]:
#Q3 - Which product categories are associated with lower purchase history, and which ones appear most among high-frequency, high-tenure customers?
# Thresholds calibrated against actual state-level distributions:
#   CLV:   median=719, std=79  → high threshold = 750 (top ~40% of states)
#   Promo: median=43%, std=6%  → low threshold  = 43% (below median)
#   Loyal: median=10%, std=3%  → high threshold = 11% (above median)
#
# Market types:
#   Organic Traction  — high CLV + low promo + above-median loyal share
#   Discount-Driven   — high CLV but high promo dependency
#   Under-Leveraged   — below-median CLV + low promo (authentic, untargeted)
#   Developing        — mixed signals
Q3 = """
SELECT
    location                                            AS state,
    COUNT(*)                                            AS n_customers,
    ROUND(AVG(purchase_amount), 2)                      AS avg_spend,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(promo_dependency_score) * 100, 1)         AS pct_promo,
    ROUND(AVG(is_loyal) * 100, 2)                       AS pct_loyal,
    ROUND(AVG(subscription_status_flag) * 100, 1)       AS pct_subscribed,
    ROUND(AVG(satisfaction_flag) * 100, 1)              AS pct_satisfied,
    CASE
        WHEN AVG(clv_proxy)              > 750
         AND AVG(promo_dependency_score) < 0.43
         AND AVG(is_loyal)               > 0.11   THEN 'Organic Traction'
        WHEN AVG(clv_proxy)              > 750
         AND AVG(promo_dependency_score) >= 0.43  THEN 'Discount-Driven'
        WHEN AVG(clv_proxy)              <= 640
         AND AVG(promo_dependency_score) < 0.40   THEN 'Under-Leveraged'
        ELSE                                           'Developing'
    END                                                 AS market_type,
    CASE
        WHEN AVG(clv_proxy) > 750
         AND AVG(promo_dependency_score) < 0.43
         AND AVG(is_loyal) > 0.11
            THEN 'High CLV + promo below median + loyal share above median'
        WHEN AVG(clv_proxy) > 750
         AND AVG(promo_dependency_score) >= 0.43
            THEN 'High CLV offset by above-median promo dependency'
        WHEN AVG(clv_proxy) <= 640
         AND AVG(promo_dependency_score) < 0.40
            THEN 'Below-median CLV despite low promo use — untapped demand'
        ELSE 'Mixed signals — no dominant pattern'
    END                                                 AS trigger_signal
FROM customers
GROUP BY location
HAVING COUNT(*) >= 50
ORDER BY avg_clv DESC
"""
 
q3 = query(Q3)
print("Q3 — GEOGRAPHIC OPPORTUNITY MAP")
print(q3.to_string(index=False))
 
q3_summary = q3.groupby("market_type").agg(
    states=("state","count"),
    avg_clv=("avg_clv","mean"),
    avg_pct_promo=("pct_promo","mean"),
    avg_pct_loyal=("pct_loyal","mean")
).round(2).sort_values("avg_clv", ascending=False)
print("Q3 — MARKET TYPE SUMMARY")
print(q3_summary.to_string())
q3.to_csv(f"{OUTPUT_DIR}/q3_geographic_opportunity.csv", index=False)
 

Q3 — GEOGRAPHIC OPPORTUNITY MAP
         state  n_customers  avg_spend  avg_clv  pct_promo  pct_loyal  pct_subscribed  pct_satisfied      market_type                                           trigger_signal
  Pennsylvania           74      66.57   932.89       44.6       9.46            28.4           41.9  Discount-Driven         High CLV offset by above-median promo dependency
        Alaska           72      67.60   877.87       40.3      13.89            25.0           45.8 Organic Traction High CLV + promo below median + loyal share above median
       Arizona           65      66.55   867.18       33.8      13.85            24.6           41.5 Organic Traction High CLV + promo below median + loyal share above median
      Illinois           92      61.05   855.30       40.2       7.61            23.9           32.6       Developing                      Mixed signals — no dominant pattern
        Hawaii           65      57.72   842.48       49.2      10.77            29.2        

In [ ]:
#Q4 - Are there cities or regions where the brand has strong traction that it has not yet deliberately targeted, and does that traction look different in terms of category preference, promo sensitivity, or average spend per customer?
# Each cell carries all four PS-required fields:
#   promo_action + trigger_signal + rollout_timeline + success_metric
Q4 = """
SELECT
    value_tier,
    customer_segment,
    COUNT(*)                                            AS n_customers,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(purchase_amount), 2)                      AS avg_spend,
    ROUND(AVG(previous_purchases), 1)                   AS avg_prev_purchases,
    ROUND(AVG(review_rating), 2)                        AS avg_rating,
    ROUND(AVG(satisfaction_flag) * 100, 1)              AS pct_satisfied,
    ROUND(AVG(frequency_score), 2)                      AS avg_freq_score,
    CASE
        WHEN value_tier = 'Premium' AND customer_segment = 'Brand Advocate'
            THEN 'SUNSET IMMEDIATELY'
        WHEN value_tier IN ('Premium','High')
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'PHASE OUT — 90-day pilot'
        WHEN value_tier = 'Mid'
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'MAINTAIN & MONITOR'
        WHEN value_tier = 'Low'
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'RE-EVALUATE (CAC vs CLV)'
        ELSE 'MAINTAIN — no promo exposure'
    END                                                 AS promo_action,
    CASE
        WHEN value_tier = 'Premium' AND customer_segment = 'Brand Advocate'
            THEN 'Avg freq=6.2, rating=4.47, zero promo use — buying at full price already'
        WHEN value_tier IN ('Premium','High')
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'High CLV with 100% promo use — test whether full price changes repeat rate'
        WHEN value_tier = 'Mid'
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'Mid CLV — promos may be protecting volume; insufficient evidence to cut yet'
        WHEN value_tier = 'Low'
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'Low CLV likely at or below acquisition cost; discounting loses money'
        ELSE 'No promo exposure — no action needed'
    END                                                 AS trigger_signal,
    CASE
        WHEN value_tier = 'Premium' AND customer_segment = 'Brand Advocate'
            THEN 'Immediate — next campaign cycle'
        WHEN value_tier IN ('Premium','High')
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN '90-day pilot — review at Day 30, 60, 90'
        WHEN value_tier = 'Mid'
            THEN 'Q3 review — collect 90 days of baseline data first'
        WHEN value_tier = 'Low'
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'Begin CAC audit in 30 days; wind down promos over 60 days if CLV < CAC'
        ELSE 'No timeline — no action needed'
    END                                                 AS rollout_timeline,
    CASE
        WHEN value_tier = 'Premium' AND customer_segment = 'Brand Advocate'
            THEN 'Repeat purchase rate within 60 days (target: no change from baseline)'
        WHEN value_tier IN ('Premium','High')
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'Repeat purchase rate + avg order value at Day 90 vs. control group'
        WHEN value_tier = 'Mid'
            THEN 'Promo revenue % of total — flag if share exceeds 50%'
        WHEN value_tier = 'Low'
         AND customer_segment IN ('Subscription Member','Promo Hunter')
            THEN 'CLV:CAC ratio — stop promos if ratio stays below 1.5x'
        ELSE 'Monitor segment migration — watch for downgrade to Promo Hunter'
    END                                                 AS success_metric
FROM customers
GROUP BY value_tier, customer_segment
ORDER BY
    CASE value_tier
        WHEN 'Premium' THEN 1 WHEN 'High' THEN 2
        WHEN 'Mid' THEN 3 ELSE 4
    END,
    avg_clv DESC
"""
 
Q4b = """
SELECT
    promo_action,
    COUNT(*)                                            AS n_customers,
    ROUND(SUM(clv_proxy), 2)                            AS total_clv_at_stake,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(purchase_amount), 2)                      AS avg_spend,
    ROUND(AVG(promo_dependency_score) * 100, 1)         AS pct_promo
FROM (
    SELECT *,
        CASE
            WHEN value_tier = 'Premium' AND customer_segment = 'Brand Advocate'
                THEN 'SUNSET IMMEDIATELY'
            WHEN value_tier IN ('Premium','High')
             AND customer_segment IN ('Subscription Member','Promo Hunter')
                THEN 'PHASE OUT — 90-day pilot'
            WHEN value_tier = 'Mid'
             AND customer_segment IN ('Subscription Member','Promo Hunter')
                THEN 'MAINTAIN & MONITOR'
            WHEN value_tier = 'Low'
             AND customer_segment IN ('Subscription Member','Promo Hunter')
                THEN 'RE-EVALUATE (CAC vs CLV)'
            ELSE 'MAINTAIN — no promo exposure'
        END AS promo_action
    FROM customers
) sub
GROUP BY promo_action
ORDER BY total_clv_at_stake DESC
"""
 
q4  = query(Q4)
q4b = query(Q4b)
print("Q4 — PROMOTIONAL RESTRUCTURE MATRIX")
print(q4.to_string(index=False))
print("Q4b — PROMO ACTION SUMMARY (CLV AT STAKE)")
print(q4b.to_string(index=False))
q4.to_csv(f"{OUTPUT_DIR}/q4_promo_restructure_matrix.csv",  index=False)
q4b.to_csv(f"{OUTPUT_DIR}/q4b_promo_action_summary.csv",    index=False)
 

Q4 — PROMOTIONAL RESTRUCTURE MATRIX
value_tier    customer_segment  n_customers  avg_clv  avg_spend  avg_prev_purchases  avg_rating  pct_satisfied  avg_freq_score                 promo_action                                                              trigger_signal                                                       rollout_timeline                                                        success_metric
   Premium      Brand Advocate          205  1567.55      75.69                32.0        4.47          100.0            6.16           SUNSET IMMEDIATELY    Avg freq=6.2, rating=4.47, zero promo use — buying at full price already                                        Immediate — next campaign cycle Repeat purchase rate within 60 days (target: no change from baseline)
   Premium        Promo Hunter          148  1478.97      77.17                32.3        3.66           37.8            5.76     PHASE OUT — 90-day pilot  High CLV with 100% promo use — test whether full price change

In [ ]:
#Q5 - What does the brand's best customer actually look like in terms of age, purchase habits, payment preferences, and satisfaction?
#ideal customer profile (ICP) = Premium + Brand Advocate
Q5a = """
SELECT 'Premium - All' AS cohort,
    COUNT(*)                                            AS n,
    ROUND(AVG(age), 1)                                  AS avg_age,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(purchase_amount), 2)                      AS avg_spend,
    ROUND(AVG(previous_purchases), 1)                   AS avg_prev_purchases,
    ROUND(AVG(frequency_score), 2)                      AS avg_freq_score,
    ROUND(AVG(review_rating), 2)                        AS avg_rating,
    ROUND(AVG(promo_dependency_score) * 100, 1)         AS pct_promo,
    ROUND(AVG(subscription_status_flag) * 100, 1)       AS pct_subscribed
FROM customers WHERE value_tier = 'Premium'
UNION ALL
SELECT 'Premium + Brand Advocate (Pure ICP)',
    COUNT(*), ROUND(AVG(age),1), ROUND(AVG(clv_proxy),2),
    ROUND(AVG(purchase_amount),2), ROUND(AVG(previous_purchases),1),
    ROUND(AVG(frequency_score),2), ROUND(AVG(review_rating),2),
    ROUND(AVG(promo_dependency_score)*100,1),
    ROUND(AVG(subscription_status_flag)*100,1)
FROM customers WHERE value_tier = 'Premium' AND customer_segment = 'Brand Advocate'
"""
 
Q5b = """
SELECT 'Gender' AS dimension, gender AS value,
    COUNT(*) AS n, ROUND(AVG(clv_proxy),2) AS avg_clv,
    ROUND(AVG(promo_dependency_score)*100,1) AS pct_promo
FROM customers WHERE value_tier = 'Premium' GROUP BY gender
UNION ALL
SELECT 'Age Group', age_group,
    COUNT(*), ROUND(AVG(clv_proxy),2), ROUND(AVG(promo_dependency_score)*100,1)
FROM customers WHERE value_tier = 'Premium' GROUP BY age_group
UNION ALL
SELECT 'Category', category,
    COUNT(*), ROUND(AVG(clv_proxy),2), ROUND(AVG(promo_dependency_score)*100,1)
FROM customers WHERE value_tier = 'Premium' GROUP BY category
UNION ALL
SELECT 'Purchase Frequency', frequency_of_purchases,
    COUNT(*), ROUND(AVG(clv_proxy),2), ROUND(AVG(promo_dependency_score)*100,1)
FROM customers WHERE value_tier = 'Premium' GROUP BY frequency_of_purchases
UNION ALL
SELECT 'Payment Method', payment_method,
    COUNT(*), ROUND(AVG(clv_proxy),2), ROUND(AVG(promo_dependency_score)*100,1)
FROM customers WHERE value_tier = 'Premium' GROUP BY payment_method
UNION ALL
SELECT 'Season', season,
    COUNT(*), ROUND(AVG(clv_proxy),2), ROUND(AVG(promo_dependency_score)*100,1)
FROM customers WHERE value_tier = 'Premium' GROUP BY season
ORDER BY dimension, avg_clv DESC
"""
 
Q5c = """
SELECT
    age_group, gender, category, season,
    COUNT(*)                                            AS n,
    ROUND(AVG(clv_proxy), 2)                            AS avg_clv,
    ROUND(AVG(purchase_amount), 2)                      AS avg_spend,
    ROUND(AVG(previous_purchases), 1)                   AS avg_prev,
    ROUND(AVG(frequency_score), 2)                      AS avg_freq,
    ROUND(AVG(review_rating), 2)                        AS avg_rating
FROM customers
WHERE value_tier = 'Premium' AND customer_segment = 'Brand Advocate'
GROUP BY age_group, gender, category, season
HAVING COUNT(*) >= 2
ORDER BY avg_clv DESC
LIMIT 20
"""
 
q5a = query(Q5a)
q5b = query(Q5b)
q5c = query(Q5c)
 
print("Q5a — PREMIUM TIER vs. PURE ICP COHORT COMPARISON")
print(q5a.to_string(index=False))
print("Q5b — ICP DIMENSIONAL BREAKDOWN (Premium Tier)")
print(q5b.to_string(index=False))
print("Q5c — PURE ICP TOP PROFILES (Premium + Brand Advocate)")
print(q5c.to_string(index=False))
q5a.to_csv(f"{OUTPUT_DIR}/q5a_icp_cohort_comparison.csv",    index=False)
q5b.to_csv(f"{OUTPUT_DIR}/q5b_icp_dimensional_profile.csv",  index=False)
q5c.to_csv(f"{OUTPUT_DIR}/q5c_icp_pure_top_profiles.csv",    index=False)
 

Q5a — PREMIUM TIER vs. PURE ICP COHORT COMPARISON
                             cohort   n  avg_age  avg_clv  avg_spend  avg_prev_purchases  avg_freq_score  avg_rating  pct_promo  pct_subscribed
                      Premium - All 974     44.5  1484.30      77.55                31.7            5.77        3.77       44.1            29.0
Premium + Brand Advocate (Pure ICP) 205     43.3  1567.55      75.69                32.0            6.16        4.47        0.0             0.0
Q5b — ICP DIMENSIONAL BREAKDOWN (Premium Tier)
         dimension          value   n  avg_clv  pct_promo
         Age Group          45-54 190  1533.18       46.3
         Age Group          18-24 115  1493.24       37.4
         Age Group          35-44 173  1480.01       43.4
         Age Group          55-70 304  1467.38       45.1
         Age Group          25-34 192  1461.24       45.3
          Category    Accessories 302  1499.92       43.7
          Category       Footwear 163  1481.84       47.9
       


IDEAL CUSTOMER PROFILE —  SUMMARY:
  Behavioral signals (primary — use for targeting and acquisition):
    Purchase frequency : Weekly or Bi-Weekly
    Previous purchases : 30+ (established relationship)
    Promo usage        : None — full-price buyer
    Review rating      : 4.0 or above
 
  Demographic signals (secondary — use for lookalike audience creation):
    Age  : 45-54 shows highest avg CLV ($1,533); 18-24 Female also strong
    Gender : Female Premium customers are 100% promo-free
    Category : Accessories and Clothing dominate the Pure ICP
    Season : No dominant season — this customer shops year-round
 
  What NOT to target on: Payment method and location show negligible CLV
  differentiation within the Premium tier. Age alone is not predictive.
  The ICP is behaviorally defined, not demographically.

In [36]:

MASTER = """
SELECT
    customer_segment,
    n_customers,
    ROUND(n_customers * 100.0 / SUM(n_customers) OVER (), 1)     AS pct_of_base,
    avg_clv,
    ROUND(total_clv * 100.0 / SUM(total_clv) OVER (), 1)         AS pct_total_clv,
    avg_spend,
    avg_prev_purchases,
    avg_rating,
    pct_promo,
    pct_satisfied,
    avg_freq_score
FROM (
    SELECT
        customer_segment,
        COUNT(*)                                                   AS n_customers,
        ROUND(AVG(clv_proxy), 2)                                   AS avg_clv,
        ROUND(SUM(clv_proxy), 2)                                   AS total_clv,
        ROUND(AVG(purchase_amount), 2)                             AS avg_spend,
        ROUND(AVG(previous_purchases), 1)                          AS avg_prev_purchases,
        ROUND(AVG(review_rating), 2)                               AS avg_rating,
        ROUND(AVG(promo_dependency_score) * 100, 1)                AS pct_promo,
        ROUND(AVG(satisfaction_flag) * 100, 1)                     AS pct_satisfied,
        ROUND(AVG(frequency_score), 2)                             AS avg_freq_score
    FROM customers
    GROUP BY customer_segment
) sub
ORDER BY avg_clv DESC
"""
 
master = query(MASTER)
print("MASTER SEGMENT SUMMARY")
print(master.to_string(index=False))
master.to_csv(f"{OUTPUT_DIR}/master_segment_summary.csv", index=False)
 

MASTER SEGMENT SUMMARY
   customer_segment  n_customers  pct_of_base  avg_clv  pct_total_clv  avg_spend  avg_prev_purchases  avg_rating  pct_promo  pct_satisfied  avg_freq_score
     Brand Advocate          393         10.1  1111.15           15.6      60.04                25.7        4.46        0.0          100.0            5.99
Subscription Member         1053         27.0   733.13           27.6      59.49                26.1        3.75      100.0           41.5            3.99
       Promo Hunter          624         16.0   694.69           15.5      58.92                25.2        3.73      100.0           40.9            3.92
      Passive Buyer         1830         46.9   629.36           41.2      60.15                24.9        3.61        0.0           30.0            3.51


In [37]:
df.to_csv(f"{OUTPUT_DIR}/customers_enriched.csv", index=False)
 
con.close()

In [38]:
outputs = [
    "customers_enriched.csv",
    "q1_loyalty_archetypes.csv",
    "q2a_value_tier_profiles.csv",
    "q2b_rfm_vs_value_tier.csv",
    "q2c_frequency_vs_clv.csv",
    "q2d_season_category_tenure.csv",
    "q2e_frequency_tenure_interaction.csv",
    "q3_geographic_opportunity.csv",
    "q4_promo_restructure_matrix.csv",
    "q4b_promo_action_summary.csv",
    "q5a_icp_cohort_comparison.csv",
    "q5b_icp_dimensional_profile.csv",
    "q5c_icp_pure_top_profiles.csv",
    "master_segment_summary.csv",
]